In [1]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU device: {torch.cuda.get_device_name(0)}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

CUDA available: True
CUDA version: 12.1
PyTorch version: 2.5.1+cu121
GPU device: NVIDIA GeForce RTX 4060 Laptop GPU
Number of GPUs: 1


In [1]:
import logging

from dotenv import load_dotenv

from rag.ingestion.document_fetcher import DocumentFetcher
from utils.data_helpers import (
    initialize_metadata_data,
    initialize_stock_data,
)

load_dotenv()

logger = logging.getLogger(__name__)


await initialize_stock_data()
await initialize_metadata_data()

c:\Users\Anant\Downloads\embed-documents-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-08 20:14:10,251 - utils.data_helpers - INFO - Initializing stock data mappings...
2026-02-08 20:14:10,295 - utils.api_utils - INFO - Retrieved active equity and sub listing data from cache
2026-02-08 20:14:10,324 - utils.data_helpers - INFO - Stock data initialized: 5517 stocks, 5501 symbols
2026-02-08 20:14:10,325 - utils.data_helpers - INFO - Initializing metadata mappings (20 years of data)...
2026-02-08 20:14:10,371 - utils.api_utils - INFO - Retrieved meta data file for 2006-02-13 to 2026-02-08 from cache
2026-02-08 20:14:10,394 - utils.data_helpers - INFO - Metadata initialized: 22109 items, 4064 unique fincodes


### Get Stocks from Nifty 50 Index

In [2]:
fetcher = DocumentFetcher()

In [3]:
test_fincode = 100325  # Reliance Industries Ltd.
docs = await fetcher.get_available_documents(
    fincode=test_fincode,
)
for doc in docs:
    logger.info(f"Document: {doc.filename} ({doc.category}) - {doc.document_date}")

2026-02-08 20:14:27,002 - rag.ingestion.document_fetcher - INFO - Fetching document metadata: fincode=100325, category=None, date_range=2024-02-09 to 2026-02-08
2026-02-08 20:14:29,014 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getMetaDataRelatedToFile?startDate=2024-02-09&endDate=2026-02-08 "HTTP/1.1 200 OK"
2026-02-08 20:14:29,210 - rag.ingestion.document_fetcher - INFO - Found 13 documents matching filters
2026-02-08 20:14:29,214 - __main__ - INFO - Document: f2454c0f-aad9-4d23-add0-dfe3d49aa8d6.pdf (concall) - 2025-10-19
2026-02-08 20:14:29,215 - __main__ - INFO - Document: b74bb5fb-c15b-42f1-b853-34231e587f46.pdf (investor-presentation) - 2025-10-17
2026-02-08 20:14:29,217 - __main__ - INFO - Document: 7a050d55-ad53-4d57-8735-cbcf20bb412b.pdf (investor-presentation) - 2025-10-17
2026-02-08 20:14:29,218 - __main__ - INFO - Document: 38a33049-ed9e-4544-ace5-a92e5f984f23.pdf (concall) - 2025-07-20
2026-02-08 20:14:29,219 - __main__ - INFO - 

In [4]:
from rag.ingestion.vector_store import QdrantManager

qdrant_manager = QdrantManager() 

2026-02-08 20:15:03,580 - rag.ingestion.document_tracker - INFO - Initialized DocumentTracker with database at c:\Users\Anant\Downloads\embed-documents-main\embed-documents-main\rag\data\embedded_docs.db
2026-02-08 20:15:03,581 - rag.ingestion.vector_store - INFO - Initialized QdrantManager with collection 'company_files'


In [5]:
exists = await qdrant_manager.check_documents_exist([doc.filename for doc in docs])
exists

{'f2454c0f-aad9-4d23-add0-dfe3d49aa8d6.pdf': True,
 'b74bb5fb-c15b-42f1-b853-34231e587f46.pdf': True,
 '7a050d55-ad53-4d57-8735-cbcf20bb412b.pdf': True,
 '38a33049-ed9e-4544-ace5-a92e5f984f23.pdf': True,
 '49a01d99-854c-4c3a-b62c-e1fb3619029f.pdf': True,
 'b6203612-be21-4d28-ace9-6496cd3fb8b2.pdf': True,
 'ac3d5e8f-9a84-4510-99e8-71eabcf49a01.pdf': True,
 'aa0547ba-09ef-404d-b57d-c216d1513f17.pdf': True,
 'cb5da39b-b46a-45f7-b2b6-932a4f2ecf91.pdf': True,
 '0e6309fa-450f-4db6-88b2-d76d9a3ff756.pdf': True,
 'fd1fe004-0eb0-4a2b-9859-7b62a89e451c.pdf': True,
 '06b05d06-8c38-4aee-aa5b-5e7108ae3871.pdf': True,
 '0501df4e-7d64-455d-bbb0-4b2118067e51.pdf': True}

In [6]:
from langchain_core.documents import Document

from rag.ingestion.llama_parse_processor import process_document

processed_docs: list[Document] = []

for doc in docs:
    if not exists[doc.filename]:
        try:
            logger.info(
                f"Document {doc.filename} does not exist in vector store. Proceeding to fetch and save."
            )
            pdf = await fetcher.get_pdf_doc_stream(doc.filename, doc.category)

            file_size = len(pdf.stream.getbuffer()) / 1024 / 1024

            logger.info(f"Document size: {file_size:,} bytes ({file_size:.2f} MB)")

            processed_docs.extend(await process_document(pdf, file_type="pdf"))
        except Exception as e:
            logger.error(f"Error processing document {doc.filename}: {e}")
            continue

In [7]:
docs_to_embed: list[Document] = []
for doc in processed_docs:
    if not exists[doc.metadata.get("source", "")]:
        docs_to_embed.append(doc)

for doc in docs_to_embed:
    logger.info(f"Document to embed: {doc.metadata.get('source', '')}")

In [8]:
if isinstance(docs_to_embed, list) and len(docs_to_embed) > 0:
    result = await qdrant_manager.embed_documents(docs_to_embed)
    logger.info(f"Cost: ${result['estimated_cost_usd']}")